In [2]:
import json
import pandas as pd
import matplotlib.pyplot as plt

In [8]:

with open("data.json", encoding="utf-8") as f:
    raw = json.load(f)

df = pd.DataFrame([
    {"nivel_m": float(nivel.removesuffix("m").replace("_", ".")),
     "bairro": bairro, **valores}
    for nivel, bairros in raw.items()
    for bairro, valores in bairros.items()
])
print(df.shape)  # deve dar (6674, 7)


(6674, 7)


In [ ]:
inconsistencias_totais = []

bairros_unicos = df["bairro"].unique()

#Passa por cada bairro um por um
for bairro in bairros_unicos:
    
    # Filtra apenas os dados do bairro atual
    df_bairro = df[df["bairro"] == bairro]
    
    # Ordena do menor nível para o maior nível do rio
    df_bairro = df_bairro.sort_values("nivel_m")
    
    # Transforma as colunas em listas do Python para facilitar a leitura
    niveis = df_bairro["nivel_m"].tolist()
    predios = df_bairro["affected_buildings"].tolist()
    
    # Compara cada nível com o nível anterior
    for i in range(1, len(predios)):
        predios_atual = predios[i]
        predios_anterior = predios[i - 1]
        
        # Se o número de prédios afetados diminuiu com o rio subindo, temos um erro
        if predios_atual < predios_anterior:
            inconsistencias_totais.append({
                "bairro": bairro,
                "nivel_m": niveis[i],
                "affected_buildings": predios_atual,
                "predios_nivel_anterior": predios_anterior
            })

# Transformamos o resultado final em um DataFrame
df_inconsistencias_fisicas = pd.DataFrame(inconsistencias_totais)

# Mostramos o resultado na tela
print(f"Total de inconsistências encontradas: {len(df_inconsistencias_fisicas)}")

if len(df_inconsistencias_fisicas) > 0:
    display(df_inconsistencias_fisicas.head(len(df_inconsistencias_fisicas)))
else:
    print("Nenhuma inconsistência encontrada.")

Total de inconsistências físicas encontradas: 12


,bairro,nivel_m,affected_buildings,predios_nivel_anterior
0,Arquipélago,3.1,4680,4685
1,Arquipélago,3.6,4680,4685
2,Arquipélago,4.1,4674,4685
3,Arquipélago,4.6,4680,4685
4,Arquipélago,5.1,4680,4685
5,Arquipélago,5.4,4680,4688
6,Centro Histórico,9.7,0,2050
7,Espírito Santo,3.4,633,637
8,Espírito Santo,3.5,25,633
9,Espírito Santo,4.8,47,2101


In [20]:
#Correção das incosistencias
df_limpo = df.copy()

# Ordena o dataframe por bairro e nível do rio para garantir a sequência correta
df_limpo = df_limpo.sort_values(by=["bairro", "nivel_m"]).reset_index(drop=True)

# Garante que o número de prédios afetados nunca diminui conforme o nível do rio sobe 
df_limpo["affected_buildings"] = df_limpo.groupby("bairro")["affected_buildings"].cummax()

# atualiza os campos dependentes (percentage e affected_population) para manter a coerência matemática
df_limpo["percentage"] = (df_limpo["affected_buildings"] / df_limpo["total_buildings"]) * 100
df_limpo["affected_population"] = df_limpo["population"] * df_limpo["percentage"] / 100

print(df_limpo.shape)

(6674, 7)


In [26]:

df_copia = df.copy()

# Recalcula a porcentagem e a população afetada baseada nas fórmulas do documento
df_copia["calc_percentage"] = (df_copia["affected_buildings"] / df_copia["total_buildings"]) * 100
df_copia["calc_pop_affected"] = df_copia["population"] * df_copia["percentage"] / 100

# Verifica onde a diferença entre o dado original e o calculado é maior que uma margem pequena de arredondamento
erro_perc = (df_copia["percentage"] - df_copia["calc_percentage"]).abs() > 0.01
erros_encontrados = df_copia[erro_perc]

print(f"Total de registros com divergências")
if len(erros_encontrados) > 0:
    display(erros_encontrados.head(10))
else:
    print(" Todos os cálculos de porcentagem estão consistentes")

Total de registros com divergências


,nivel_m,bairro,affected_buildings,total_buildings,percentage,population,affected_population,calc_percentage,calc_pop_affected
2,3.0,Anchieta,28,589,5,791,40,4.753820,39.55
3,3.0,Arquipélago,4685,4688,100,6411,6411,99.936007,6411.00
7,3.0,Belém Novo,95,6012,2,9851,198,1.580173,197.02
10,3.0,Boa Vista do Sul,306,2450,13,2703,352,12.489796,351.39
17,3.0,Centro Histórico,42,4312,1,30569,306,0.974026,305.69
23,3.0,Cristal,20,10229,1,24851,249,0.195523,248.51
25,3.0,Espírito Santo,637,3342,20,4953,991,19.060443,990.60
27,3.0,Farrapos,4377,5510,80,17591,14073,79.437387,14072.80
29,3.0,Floresta,20,3731,1,8798,88,0.536049,87.98
31,3.0,Guarujá,240,2912,9,6145,554,8.241758,553.05
